# Lumbar Spinal Stenosis Classification & Spot Localization

This notebook implements a deep learning pipeline using PyTorch to detect and classify **Lumbar Spinal Stenosis (LSS)** and visualize the specific regions of stenosis (such as herniated discs or thecal sac compression) using **Grad-CAM (Gradient-weighted Class Activation Mapping)**.

## Project Structure
- **Dataset**: Lumbar spine MRI scans containing 13,686 images across 3 classes:
  - `Herniated Disc`: Vertebral disc herniation compressing nerve roots.
  - `Thecal Sac`: Compression of the thecal sac.
  - `No Stenosis`: Scans with healthy canal spacing.
- **Model**: Pre-trained ResNet-50 model adapted for 3-class classification.
- **Explainability**: Custom PyTorch implementation of **Grad-CAM** to highlight where the model is looking in the spinal scan.

## Technical Challenge: Grayscale Images
Since the spinal MRI scans are preprocessed to $400 \times 400 \times 1$ (Grayscale), we will load them and convert them to 3-channel RGB format (by duplicating the single channel). This allows us to leverage the pre-trained weights of ResNet-50.

## How Grad-CAM Works
Grad-CAM computes the gradients of the target class prediction score with respect to the feature map activations of the final convolutional layer (`model.layer4`). By global-average-pooling these gradients and taking a weighted combination of activations, we obtain a heatmap indicating which regions of the lower back (L1-L5 vertebrae) drove the classification.

In [ ]:
import os
import random
import numpy as np
import cv2
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
import torchvision
from torchvision import transforms, datasets
from torchvision.models import resnet50, ResNet50_Weights

from sklearn.metrics import classification_report, confusion_matrix

# Set seed for reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

# Check device (CUDA GPU, MPS for Mac, or CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")


## Dataset Verification & Exploration

Let's locate the dataset directory and print the number of samples in the `train` and `test` folders. This ensures that the data paths are correct and maps out the distribution of our classes. We will also display sample spinal MRI images from each category.

In [ ]:
# Configure dataset path
LOCAL_PATH = "./data/Lumbar Spinal Stenosis data/LumbarSpinalStenosis"
# Kaggle dataset path
KAGGLE_PATH = "/kaggle/input/diagnosis-lumbar-spinal-patients-mri-dataset/LumbarSpinalStenosis"

DATA_DIR = LOCAL_PATH if os.path.exists(LOCAL_PATH) else KAGGLE_PATH
print(f"Targeting dataset directory: {DATA_DIR}")

train_dir = os.path.join(DATA_DIR, "train")
test_dir = os.path.join(DATA_DIR, "test")

# Print distribution
for split, path in [("train", train_dir), ("test", test_dir)]:
    if os.path.exists(path):
        print(f"\n--- {split} Split ---")
        categories = sorted(os.listdir(path))
        for cat in categories:
            cat_path = os.path.join(path, cat)
            if os.path.isdir(cat_path):
                num_images = len([f for f in os.listdir(cat_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
                print(f"Category '{cat}': {num_images} images")
    else:
        print(f"\n{split} path does not exist at {path}")

# Plot one sample image from each class
if os.path.exists(train_dir):
    categories = sorted(os.listdir(train_dir))
    fig, axes = plt.subplots(1, len(categories), figsize=(15, 5))
    for i, cat in enumerate(categories):
        cat_path = os.path.join(train_dir, cat)
        img_files = [f for f in os.listdir(cat_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
        if img_files:
            sample_img_path = os.path.join(cat_path, img_files[0])
            img = Image.open(sample_img_path)
            axes[i].imshow(img, cmap='gray')
            axes[i].set_title(f"{cat}\nShape: {img.size}")
            axes[i].axis('off')
    plt.tight_layout()
    plt.show()


## Data Pipelines, Preprocessing & Augmentation

To prepare the 1-channel (grayscale) images for transfer learning:
1. **RGB Conversion**: We will load all images as PIL RGB images (which copies the single channel to three identical channels).
2. **Training Transforms**: Resize to $224 \times 224$, apply Random Rotation ($10^{\circ}$), Random Horizontal Flip, convert to Tensor, and normalize with ImageNet mean and std.
3. **Validation & Test Transforms**: Standard resizing to $224 \times 224$, convert to Tensor, and normalize (no random augmentations).

We will split the `train` folder into **Train** ($85\%$) and **Validation** ($15\%$) sets to track the loss and accuracy metrics during training. The `test` folder remains untouched for final evaluation.

In [ ]:
# Define parameters
BATCH_SIZE = 32
IMAGE_SIZE = 224

# Data normalization statistics for pretrained ResNet-50
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

# Data Transformations
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomRotation(10),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

val_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

# Custom Wrapper to handle subset mapping and RGB conversion
class DatasetWrapper(torch.utils.data.Dataset):
    def __init__(self, subset, transform=None):
        self.subset = subset
        self.transform = transform

    def __getitem__(self, index):
        # Retrieve the original PIL image and label from the subset
        image, label = self.subset[index]
        # Force conversion to RGB (copies 1 grayscale channel into 3 channels)
        image = image.convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, label

    def __len__(self):
        return len(self.subset)

# Load full training folder without transforms first
if os.path.exists(train_dir):
    full_train_dataset = datasets.ImageFolder(train_dir, transform=None)
    classes = full_train_dataset.classes
    print(f"Detected classes: {classes}")
    
    # Split into Train (85%) and Validation (15%)
    train_len = int(0.85 * len(full_train_dataset))
    val_len = len(full_train_dataset) - train_len
    
    train_subset, val_subset = random_split(
        full_train_dataset, 
        [train_len, val_len],
        generator=torch.Generator().manual_seed(42)
    )
    
    # Wrap subsets with respective transformations
    train_dataset = DatasetWrapper(train_subset, transform=train_transform)
    val_dataset = DatasetWrapper(val_subset, transform=val_transform)
    
    # Create Dataloaders
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
    
    print(f"Train subset size: {len(train_dataset)}")
    print(f"Validation subset size: {len(val_dataset)}")
else:
    print(f"ERROR: Train directory not found at {train_dir}")

# Load test dataset with RGB wrapper
if os.path.exists(test_dir):
    full_test_dataset = datasets.ImageFolder(test_dir, transform=None)
    test_dataset = DatasetWrapper(full_test_dataset, transform=val_transform)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
    print(f"Test dataset size: {len(test_dataset)}")
else:
    print(f"ERROR: Test directory not found at {test_dir}")


## Model Architecture: Transfer Learning with ResNet-50

We will load a pre-trained **ResNet-50** architecture.
- We will replace the final classification head (`model.fc`), which has 2048 input features, with a new linear layer mapping to **3 output classes** (Herniated Disc, Thecal Sac, and No Stenosis).
- We will fine-tune the entire model with a learning rate of $10^{-4}$ to allow the network to adjust its features to the vertical structures of the human spine.

In [ ]:
def initialize_model(num_classes=3):
    # Load ResNet-50 model with default pre-trained weights
    model = resnet50(weights=ResNet50_Weights.DEFAULT)
    
    # Replace the final fully connected layer (classifier)
    num_ftrs = model.fc.in_features
    model.fc = nn.Linear(num_ftrs, num_classes)
    
    return model

# Initialize the model and move it to the device
model = initialize_model(num_classes=3)
model = model.to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Model: ResNet-50 initialized.")
print(f"Total Parameters: {total_params:,}")
print(f"Trainable Parameters: {trainable_params:,}")


## Explainable AI: Grad-CAM for Spine MRI Scan Analysis

We will define our custom **Grad-CAM** module to extract feature maps and gradients from the final convolutional block of ResNet-50 (`model.layer4`). 

We will generate heatmaps and overlay them on the original spinal MRIs, helping visualize the exact compression points of the herniated disc or thecal sac.

In [ ]:
class GradCAM:
    """
    Custom Grad-CAM implementation in PyTorch.
    Registers forward and backward hooks on the last convolutional layer.
    """
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        
        # Register hooks
        self.forward_hook = self.target_layer.register_forward_hook(self.save_activation)
        
        # Use full backward hook if available (for PyTorch >= 1.8 compatibility)
        try:
            self.backward_hook = self.target_layer.register_full_backward_hook(self.save_gradient)
        except AttributeError:
            self.backward_hook = self.target_layer.register_backward_hook(self.save_gradient)
            
    def save_activation(self, module, input, output):
        self.activations = output.detach()
        
    def save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()
        
    def __call__(self, input_tensor, class_idx=None):
        self.model.zero_grad()
        
        # Forward pass
        output = self.model(input_tensor)
        
        if class_idx is None:
            class_idx = torch.argmax(output, dim=1).item()
            
        # Backward pass for the target class
        loss = output[0, class_idx]
        loss.backward()
        
        # Retrieve gradients and activations
        gradients = self.gradients[0]      # Shape: [C, H, W]
        activations = self.activations[0]  # Shape: [C, H, W]
        
        # Global Average Pooling of gradients (weights alpha)
        weights = torch.mean(gradients, dim=(1, 2), keepdim=True) # Shape: [C, 1, 1]
        
        # Weighted combination of activation maps
        cam = torch.sum(weights * activations, dim=0) # Shape: [H, W]
        
        # Apply ReLU to keep only positive contributions
        cam = torch.clamp(cam, min=0)
        
        # Normalize CAM to range [0, 1]
        cam_min, cam_max = cam.min(), cam.max()
        if cam_max > cam_min:
            cam = (cam - cam_min) / (cam_max - cam_min)
        else:
            cam = cam - cam_min
            
        return cam.cpu().numpy(), class_idx
        
    def release(self):
        # Remove hooks to free up memory
        self.forward_hook.remove()
        self.backward_hook.remove()


def overlay_cam(pil_img, cam, alpha=0.4):
    """
    Overlays a Grad-CAM heatmap onto the original PIL image.
    """
    # Resize original image to standard 224x224 and convert to RGB
    raw_img = np.array(pil_img.convert('RGB').resize((224, 224)))
    
    # Scale CAM heatmap
    cam_resized = cv2.resize(cam, (224, 224))
    heatmap = np.uint8(255 * cam_resized)
    
    # Apply JET color map
    heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
    
    # Overlay heatmap with transparency
    overlaid = cv2.addWeighted(heatmap, alpha, raw_img, 1.0 - alpha, 0)
    
    # Return normalized values for matplotlib plotting
    return raw_img / 255.0, heatmap / 255.0, overlaid / 255.0


## Training Setup & Optimization

For our spine model:
1. **Loss Function**: `nn.CrossEntropyLoss()`
2. **Optimizer**: `AdamW` (learning rate $10^{-4}$, weight decay $10^{-2}$).
3. **Scheduler**: `CosineAnnealingLR` over 10 epochs.
4. **Checkpointing**: Saves the highest validation accuracy parameters to `best_spine_model.pth`.

In [ ]:
# Hyperparameters
EPOCHS = 10
LEARNING_RATE = 1e-4

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-2)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

history = {
    "train_loss": [], "train_acc": [],
    "val_loss": [], "val_acc": []
}

best_val_acc = 0.0
checkpoint_path = "best_spine_model.pth"

print("Starting training pipeline...")

for epoch in range(1, EPOCHS + 1):
    # 1. Train epoch
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        train_correct += torch.sum(preds == labels).item()
        train_total += labels.size(0)
        
    epoch_train_loss = train_loss / train_total
    epoch_train_acc = train_correct / train_total
    
    # 2. Validate epoch
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            val_correct += torch.sum(preds == labels).item()
            val_total += labels.size(0)
            
    epoch_val_loss = val_loss / val_total
    epoch_val_acc = val_correct / val_total
    
    # Step scheduler
    scheduler.step()
    
    # Log progress
    history["train_loss"].append(epoch_train_loss)
    history["train_acc"].append(epoch_train_acc)
    history["val_loss"].append(epoch_val_loss)
    history["val_acc"].append(epoch_val_acc)
    
    current_lr = scheduler.get_last_lr()[0]
    print(f"Epoch {epoch:02d}/{EPOCHS:02d} | "
          f"Train Loss: {epoch_train_loss:.4f} - Train Acc: {epoch_train_acc*100:.2f}% | "
          f"Val Loss: {epoch_val_loss:.4f} - Val Acc: {epoch_val_acc*100:.2f}% | "
          f"LR: {current_lr:.6f}")
    
    # Save best checkpoint
    if epoch_val_acc > best_val_acc:
        best_val_acc = epoch_val_acc
        torch.save(model.state_dict(), checkpoint_path)
        print(f"  --> Saved new best checkpoint with Val Acc: {best_val_acc*100:.2f}%")

print(f"\nTraining completed! Best Validation Accuracy: {best_val_acc*100:.2f}%")

# Plot training and validation curves
epochs_range = range(1, EPOCHS + 1)
plt.figure(figsize=(12, 5))

# Plot Loss
plt.subplot(1, 2, 1)
plt.plot(epochs_range, history["train_loss"], label="Train Loss", color="royalblue", marker='o')
plt.plot(epochs_range, history["val_loss"], label="Val Loss", color="orange", marker='x')
plt.title("Loss Curves")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)

# Plot Accuracy
plt.subplot(1, 2, 2)
plt.plot(epochs_range, history["train_acc"], label="Train Acc", color="royalblue", marker='o')
plt.plot(epochs_range, history["val_acc"], label="Val Acc", color="orange", marker='x')
plt.title("Accuracy Curves")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()


## Model Evaluation on Testing Split

We load our best model weights (`best_spine_model.pth`) and run inference across all images in the `test` split. We will display a classification report and plot the confusion matrix. 

*Note: In this specific dataset split on Mendeley, the testing folder for the "Thecal Sac" class contains very few images (~13). The classification report and confusion matrix will show how the model evaluates across these unbalanced test sets.*

In [ ]:
# Load the best model weights
if os.path.exists(checkpoint_path):
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    print("Loaded best model weights successfully.")

# Run predictions on the test loader
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

# Generate Classification Report
print("\n=== CLASSIFICATION REPORT ===")
print(classification_report(all_labels, all_preds, target_names=classes))

# Generate and plot Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
plt.title("Confusion Matrix (Testing Set)")
plt.xlabel("Predicted Labels")
plt.ylabel("True Labels")
plt.tight_layout()
plt.show()


## Spinal Stenosis Localization with Grad-CAM

We will now initialize the Grad-CAM module and select a random test sample from each of the 3 classes. 

This will output three images side-by-side:
1. **Original Spinal MRI**: The raw scan.
2. **Grad-CAM Heatmap**: Representing the features that contributed to the classification.
3. **Overlaid Spot Highlight**: The heatmap superimposed over the spine scan.

By doing this, we can see if the model is highlighting the exact vertebrae (L1-L5) or disc spacing where the compression is happening.

In [ ]:
# Initialize Grad-CAM targeting ResNet-50's final layer block (layer4)
grad_cam = GradCAM(model, model.layer4)

# Find one sample from each class in the test dataset dynamically
# We access test_dataset.subset.samples because test_dataset is wrapped by DatasetWrapper
samples_to_visualize = {}
for path, label in test_dataset.subset.samples:
    class_name = classes[label]
    if class_name not in samples_to_visualize:
        samples_to_visualize[class_name] = (path, label)
    if len(samples_to_visualize) == len(classes):
        break

# Run visualization
for class_name, (img_path, label) in samples_to_visualize.items():
    # 1. Load original image and convert to RGB
    pil_img = Image.open(img_path).convert('RGB')
    
    # 2. Preprocess to tensor for model forward pass
    input_tensor = val_transform(pil_img).unsqueeze(0).to(device)
    
    # 3. Generate CAM (let Grad-CAM automatically select the predicted class index)
    model.eval()
    cam, pred_idx = grad_cam(input_tensor, class_idx=None)
    
    # 4. Generate overlaid map
    raw_img, heatmap, overlaid = overlay_cam(pil_img, cam, alpha=0.45)
    
    # 5. Plot the three states side by side
    fig, axes = plt.subplots(1, 3, figsize=(14, 5))
    
    # Column 1: Original MRI
    axes[0].imshow(raw_img)
    axes[0].set_title(f"Original MRI\n(True Class: {class_name.capitalize()})", fontsize=12)
    axes[0].axis('off')
    
    # Column 2: Grad-CAM Heatmap
    axes[1].imshow(heatmap)
    axes[1].set_title("Grad-CAM Heatmap\n(Activation Focus)", fontsize=12)
    axes[1].axis('off')
    
    # Column 3: Overlaid Spot Highlight
    axes[2].imshow(overlaid)
    axes[2].set_title(f"Overlaid Spot Highlight\n(Predicted Class: {classes[pred_idx].capitalize()})", fontsize=12)
    axes[2].axis('off')
    
    plt.tight_layout()
    plt.show()

# Release hooks after visualization
grad_cam.release()
print("Grad-CAM visualization completed and hooks released.")

## Summary and Next Steps

This notebook provides a complete pipeline to train a deep learning classifier for Lumbar Spinal Stenosis and localize stenosis spots using Grad-CAM.

### Q&A
**Q**: How can I run this on the 1-channel (grayscale) Lumbar Spinal dataset using pre-trained weights?
**A**: We load the grayscale images and convert them into 3-channel RGB representation by duplicating the single channel. This permits full reuse of the pre-trained weights in the first layer (`conv1`) of ResNet-50.

### Data Analysis Key Findings
- **Class Balance**: The training split contains 10,948 images: `Herniated Disc` (3,063), `No Stenosis` (3,152), and `Thecal Sac` (4,733).
- **Test Set Imbalance**: The testing split contains 2,738 images: `Herniated Disc` (1,218), `No Stenosis` (1,507), and `Thecal Sac` (13). The validation split (1,643 images extracted from the train folder) will provide a more representative evaluation for the `Thecal Sac` category.
- **Grayscale Resolution**: The original files are preprocessed to $400 \times 400 \times 1$ and are resized to $224 \times 224$ for ResNet-50 training speed.

### Insights or Next Steps
- **Kaggle Execution**: Upload `spine_stenosis_detection.ipynb` to Kaggle and link it to the **Diagnosis Lumbar Spinal Patients MRI Dataset**.
- **GPU Accelerator**: Run the notebook with **GPU T4 x2** or **GPU P100** enabled in Kaggle settings.
- **Weights Preservation**: Best weights will be stored as `best_spine_model.pth` in the output folder, ready to be downloaded and used for local inference.